# IFLS5 — Weighted Analysis
Survey-weighted logistic regression using `pwt14xa` (person cross-sectional weight, IFLS5 2014) from `ptrack.dta`.

In [ ]:
%pip install pingouin --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)

## 1. Load Raw Data

In [ ]:
# Cover: marital status, age, sex
cov = pd.read_stata('adult_a/b3a_cov.dta')[['pidlink', 'marstat', 'age', 'sex']]

# Demographics: ethnicity, education
dl1 = pd.read_stata('adult_a/b3a_dl1.dta')[['pidlink', 'dl01f', 'dl06']]

# Income: monthly (tk25a1) and yearly (tk25a2)
tk2 = pd.read_stata('adult_a/b3a_tk2.dta')[['pidlink', 'tk25a1', 'tk25a2']]

# Sampling weights: person cross-sectional weight for IFLS5 (2014)
ptrack = pd.read_stata('tracking_book/ptrack.dta')[['pidlink', 'pwt14xa']]

print('cov   :', cov.shape)
print('dl1   :', dl1.shape)
print('tk2   :', tk2.shape)
print('ptrack:', ptrack.shape)

## 2. Merge Datasets

In [ ]:
df = cov.merge(dl1, on='pidlink', how='inner') \
        .merge(tk2, on='pidlink', how='inner') \
        .merge(ptrack, on='pidlink', how='left')

df = df.rename(columns={
    'tk25a1': 'monthly_income',
    'tk25a2': 'yearly_income',
    'dl01f' : 'ethnicity',
    'dl06'  : 'education'
})

print('Merged shape:', df.shape)
print('Weight coverage (non-null):', df['pwt14xa'].notna().sum(), '/', len(df))

## 3. Clean Variables

In [ ]:
# ── Age ──────────────────────────────────────────────────────────────
df = df[df['age'] != "998:Don't Know"]
df['age'] = df['age'].astype(float).astype(int)
df = df[df['age'] >= 18]

# ── Marital status labels ─────────────────────────────────────────────
marstat_map = {
    '1:Not yet married': 'Never Married',
    '2:Married'        : 'Married',
    '3:Separated'      : 'Separated',
    '4:Divorced'       : 'Divorced',
    '5:Widowed'        : 'Widowed',
    '6:Cohabitate'     : 'Cohabitate',
}
df['marstat'] = df['marstat'].map(marstat_map).fillna(df['marstat'])

# ── Sex labels ────────────────────────────────────────────────────────
df['sex'] = df['sex'].map({'1:Male': 'Male', '3:Female': 'Female'}).fillna(df['sex'])

# ── Ethnicity mapping ─────────────────────────────────────────────────
ethnic_map = {
    'A':'Jawa','B':'Sunda','C':'Bali','D':'Batak','E':'Bugis','F':'Tionghoa',
    'G':'Madura','H':'Sasak','I':'Minang','J':'Banjar','K':'Bima-Dompu',
    'L':'Makassar','M':'Nias','N':'Palembang','O':'Sumbawa','P':'Toraja',
    'Q':'Betawi','R':'Dayak','S':'Melayu','T':'Komering','U':'Ambon',
    'A1':'Manado','B1':'Aceh','C1':'Other South Sumatera','D1':'Banten',
    'E1':'Cirebon','F1':'Gorontalo','G1':'Kutai','V':'Other',
}
df['ethnicity'] = df['ethnicity'].map(ethnic_map).fillna('Other')

# ── Education ─────────────────────────────────────────────────────────
df['education'] = pd.to_numeric(df['education'], errors='coerce')

print('Shape after cleaning:', df.shape)
print(df[['marstat','sex','ethnicity','age']].describe(include='all'))

## 4. Handle IFLS Missing Codes & Income Transform

In [ ]:
IFLS_MISSING = [999999997, 999999998, 999999999]
df['monthly_income'] = df['monthly_income'].replace(IFLS_MISSING, np.nan)
df['yearly_income']  = df['yearly_income'].replace(IFLS_MISSING, np.nan)

# Log-transform income (log1p handles zeros)
df['log_income']  = np.log1p(df['monthly_income'])
df['age_squared'] = df['age'] ** 2

print(df[['monthly_income','log_income','age','age_squared']].describe())

## 5. Construct Outcome Variables

In [ ]:
# Ever married (1 = ever married, widowed, divorced, separated, cohabitate)
df['ever_married'] = (df['marstat'] != 'Never Married').astype(int)

# Ever divorced (1 = currently divorced or separated; used within ever-married subsample)
df['is_divorced'] = df['marstat'].isin(['Divorced', 'Separated']).astype(int)

# Ethnicity dummies (reference: all other ethnicities)
df['is_jawa']  = (df['ethnicity'] == 'Jawa').astype(int)
df['is_sunda'] = (df['ethnicity'] == 'Sunda').astype(int)

print('Ever married :', df['ever_married'].value_counts().to_dict())
print('Is divorced  :', df['is_divorced'].value_counts().to_dict())
print('Sex          :', df['sex'].value_counts().to_dict())

## 6. Final Sample — Drop Missing & Save

In [ ]:
analysis_cols = ['pidlink','sex','age','age_squared','education',
                 'monthly_income','log_income','marstat',
                 'ever_married','is_divorced','is_jawa','is_sunda','pwt14xa']

df_clean = df[analysis_cols].dropna()

print('Final sample:', df_clean.shape)
print('Male  :', (df_clean['sex']=='Male').sum())
print('Female:', (df_clean['sex']=='Female').sum())
print('Weight non-null:', df_clean['pwt14xa'].notna().sum())

df_clean.to_csv('clean_data/data_clean_weighted.csv', index=False)

## 7. Weighted Descriptive Statistics

In [ ]:
def weighted_mean(series, weights):
    mask = series.notna() & weights.notna()
    return np.average(series[mask], weights=weights[mask])

def weighted_se(series, weights):
    mask = series.notna() & weights.notna()
    w = weights[mask].values
    x = series[mask].values
    wm = np.average(x, weights=w)
    variance = np.average((x - wm)**2, weights=w)
    return np.sqrt(variance / mask.sum())

vars_desc = {
    'Age'                    : 'age',
    'Monthly Income (IDR)'   : 'monthly_income',
    'Education (years)'      : 'education',
    'Ever Married (prop)'    : 'ever_married',
    'Ever Divorced (prop)'   : 'is_divorced',
    'Javanese (prop)'        : 'is_jawa',
    'Sundanese (prop)'       : 'is_sunda',
}

rows = []
for label, col in vars_desc.items():
    for grp, gdf in [('Women', df_clean[df_clean['sex']=='Female']),
                     ('Men',   df_clean[df_clean['sex']=='Male'])]:
        rows.append({
            'Variable': label,
            'Group'   : grp,
            'Mean'    : round(weighted_mean(gdf[col], gdf['pwt14xa']), 3),
            'SE'      : round(weighted_se(gdf[col],   gdf['pwt14xa']), 3),
        })

desc_table = pd.DataFrame(rows).pivot(index='Variable', columns='Group', values=['Mean','SE'])
desc_table.columns = ['Men Mean','Men SE','Women Mean','Women SE']
print('\n=== Table 1: Weighted Means and Standard Errors ===')
print(desc_table.to_string())

## 8. Weighted Logistic Regression — Helper Functions

In [ ]:
def run_weighted_model(df_sub, outcome, formula_rhs, label):
    """
    Weighted logistic regression using GLM Binomial with survey weights (pwt14xa).
    Weights are normalized to sum to sample size (standard probability-weight approach).
    """
    df_m = df_sub.dropna(subset=[outcome] + formula_rhs.replace('~','').split('+'))
    df_m = df_m.copy()

    # Normalize weights: sum to N so model degrees-of-freedom are preserved
    n = len(df_m)
    df_m['w_norm'] = df_m['pwt14xa'] / df_m['pwt14xa'].sum() * n

    formula = f'{outcome} {formula_rhs}'
    model = smf.glm(
        formula,
        data   = df_m,
        family = sm.families.Binomial(),
        var_weights = df_m['w_norm']
    ).fit()

    print(f'\n--- {label} (N={n}) ---')
    print(model.summary2().tables[1][['Coef.','Std.Err.','z','P>|z|']].round(3))
    print(f'Pseudo R²: {1 - model.deviance / model.null_deviance:.4f}')
    return model


def format_results_table(m_female, m_male, title):
    """Side-by-side coefficient table with significance stars."""
    def fmt(coef, pval):
        stars = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else ''
        return f'{coef:.3f}{stars}'

    results = {}
    for label, model in [('Women', m_female), ('Men', m_male)]:
        t = model.summary2().tables[1]
        results[label] = {
            idx: f"{fmt(row['Coef.'], row['P>|z|'])}\n({row['Std.Err.']:.3f})"
            for idx, row in t.iterrows()
        }

    tbl = pd.DataFrame(results)
    pr2_w = 1 - m_female.deviance / m_female.null_deviance
    pr2_m = 1 - m_male.deviance   / m_male.null_deviance
    tbl.loc['Pseudo R²'] = [f'{pr2_w:.3f}', f'{pr2_m:.3f}']
    tbl.loc['N']         = [m_female.nobs, m_male.nobs]

    print(f'\n=== {title} ===')
    print(tbl.to_string())
    print('* p<0.05  ** p<0.01  *** p<0.001')
    return tbl

## 9. Model 1 — Ever Married

In [ ]:
df_male   = df_clean[df_clean['sex'] == 'Male'].copy()
df_female = df_clean[df_clean['sex'] == 'Female'].copy()

print('Male  :', len(df_male))
print('Female:', len(df_female))

formula_base = '~ log_income + education + age + age_squared + is_jawa + is_sunda'

m1_female = run_weighted_model(df_female, 'ever_married', formula_base, 'Ever Married — Women')
m1_male   = run_weighted_model(df_male,   'ever_married', formula_base, 'Ever Married — Men')

tbl1 = format_results_table(m1_female, m1_male, 'TABLE 2: EVER MARRIED')

## 10. Model 2 — Ever Divorced (Ever Married Only)

In [ ]:
df_em = df_clean[df_clean['ever_married'] == 1].copy()

df_em_male   = df_em[df_em['sex'] == 'Male'].copy()
df_em_female = df_em[df_em['sex'] == 'Female'].copy()

print('Ever married — Male  :', len(df_em_male))
print('Ever married — Female:', len(df_em_female))
print('Divorced — Male  :', df_em_male['is_divorced'].sum())
print('Divorced — Female:', df_em_female['is_divorced'].sum())

formula_divorce = '~ log_income + education + age + age_squared + is_jawa + is_sunda'

m2_female = run_weighted_model(df_em_female, 'is_divorced', formula_divorce, 'Ever Divorced — Women')
m2_male   = run_weighted_model(df_em_male,   'is_divorced', formula_divorce, 'Ever Divorced — Men')

tbl2 = format_results_table(m2_female, m2_male, 'TABLE 3: EVER DIVORCED (Ever Married Only)')

## 11. Predicted Probability Plots

In [ ]:
def plot_predicted_prob(m_female, m_male, df_female, df_male, outcome_label, filename):
    income_range = np.linspace(
        df_clean['log_income'].quantile(0.05),
        df_clean['log_income'].quantile(0.95),
        100
    )

    fig, ax = plt.subplots(figsize=(9, 5))

    for model, df_sub, color, label in [
        (m_female, df_female, '#e74c3c', 'Women'),
        (m_male,   df_male,   '#2980b9', 'Men'),
    ]:
        means = df_sub[['education','age','age_squared','is_jawa','is_sunda']].mean()
        pred_data = pd.DataFrame({
            'log_income'  : income_range,
            'education'   : means['education'],
            'age'         : means['age'],
            'age_squared' : means['age_squared'],
            'is_jawa'     : means['is_jawa'],
            'is_sunda'    : means['is_sunda'],
        })
        pred = model.predict(pred_data)
        ax.plot(income_range, pred, color=color, label=label, linewidth=2)

    ax.set_xlabel('Log Monthly Income', fontsize=12)
    ax.set_ylabel('Predicted Probability', fontsize=12)
    ax.set_title(f'Predicted Probability: {outcome_label}', fontsize=13)
    ax.legend()
    ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    plt.show()

plot_predicted_prob(m1_female, m1_male, df_female, df_male,
                   'Ever Married by Log Income', 'ever_married_weighted.png')

plot_predicted_prob(m2_female, m2_male, df_em_female, df_em_male,
                   'Ever Divorced by Log Income (Ever Married Only)', 'ever_divorced_weighted.png')

## End